# Universal Multi-Company Stock LSTM (Google Colab)
### Train ONE Unified Deep Learning Model Across All Company Datasets

This notebook allows you to:
1. **Ingest an entire folder of company CSVs** (e.g. 55 NIFTY 50 stocks: TCS, INFY, RELIANCE, HDFCBANK, etc.).
2. **Normalize each stock independently** to prevent expensive stocks (e.g. Rs 3,500) from dominating cheaper stocks (e.g. Rs 100).
3. **Train ONE universal PyTorch LSTM** with **AdamW** and **ReduceLROnPlateau** on GPU.
4. **Interact with any company stock via an interactive dropdown** to immediately see its specific multi-step forecast cone, test accuracy, and directional trend.

---
## 1. Hardware Check & Environment Setup
Ensure **GPU** is enabled: `Runtime` -> `Change runtime type` -> `T4 GPU`.

In [ ]:
# Check GPU availability
!nvidia-smi

# Install / verify required packages
!pip install -q torch pandas numpy matplotlib scikit-learn yfinance ipywidgets

---
## 2. Clone Your GitHub Repository
Clone your repository containing the code and dataset files.

In [ ]:
import os

# Clone project repository
REPO_URL = "https://github.com/Raghava-sensai/stock-lstm-prediction.git"
REPO_DIR = "stock-lstm-prediction"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

print(f"\n[OK] Current Active Directory: {os.getcwd()}")

---
## 3. Verify Dataset Folder
Check all company CSV files found in the dataset directory.

In [ ]:
import glob

DATA_DIR = "dataset/archive/NifSent/NIFTY 50"
csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
if not csv_files:
    csv_files = glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)

print(f"[*] Found {len(csv_files)} company CSV datasets in {DATA_DIR}:\n")
tickers = [os.path.splitext(os.path.basename(f))[0] for f in sorted(csv_files)]
for i in range(0, len(tickers), 6):
    print("  ".join(f"{t:<12}" for t in tickers[i:i+6]))

---
## 4. Train ONE Universal Model on All Companies (GPU)
Trains ONE unified model across all companies in the folder using AdamW and per-company independent scaling.

In [ ]:
# Train ONE unified model across all companies on GPU
!python multi_company_unified_lstm.py \
    --data_dir "dataset/archive/NifSent/NIFTY 50" \
    --epochs 20 \
    --batch_size 256 \
    --seq_length 24 \
    --forecast_horizon 5 \
    --output_dir "./outputs/universal_multi_company"

---
## 5. Multi-Company Leaderboard
Inspect model performance across all companies sorted by lowest test MAPE (Mean Absolute Percentage Error).

In [ ]:
import pandas as pd
from IPython.display import display

leaderboard_path = "outputs/universal_multi_company/multi_company_leaderboard.csv"
if os.path.exists(leaderboard_path):
    df_board = pd.read_csv(leaderboard_path)
    print(f"\n[+] Master Multi-Company Leaderboard ({len(df_board)} Companies):")
    display(df_board.head(25))
else:
    print("[!] Run training in Section 4 first to generate leaderboard.")

---
## 6. Interactive Stock Selector & Forecast Viewer
Select **any company** from the dropdown below to view its specific actual vs predicted chart and 5-candle forecast cone!

In [ ]:
import pickle
from multi_company_unified_lstm import view_company_forecast, launch_interactive_widget

cache_file = "outputs/universal_multi_company/eval_results.pkl"
with open(cache_file, "rb") as f:
    eval_results = pickle.load(f)

# Launch interactive dropdown widget
launch_interactive_widget(eval_results)

---
### Direct Form Selector (Colab Parameter Slider)
Alternatively, pick any stock ticker using the Colab form field below:

In [ ]:
#@title Select Company Stock { run: "auto" }
selected_ticker = "RELIANCE" #@param ["ADANIENT", "ADANIPORTS", "APOLLOHOSP", "ASIANPAINT", "AXISBANK", "BAJAJ-AUTO", "BAJFINANCE", "BAJAJFINSV", "BPCL", "BHARTIARTL", "BRITANNIA", "CIPLA", "COALINDIA", "DIVISLAB", "DRREDDY", "EICHERMOT", "GRASIM", "HCLTECH", "HDFCBANK", "HDFCLIFE", "HEROMOTOCO", "HINDALCO", "HINDUNILVR", "ICICIBANK", "ITC", "INDUSINDBK", "INFY", "JSWSTEEL", "KOTAKBANK", "LT", "LTIM", "M&M", "MARUTI", "NTPC", "NESTLEIND", "ONGC", "POWERGRID", "RELIANCE", "SBILIFE", "SBIN", "SUNPHARMA", "TCS", "TATACONSUM", "TATAMOTORS", "TATASTEEL", "TECHM", "TITAN", "ULTRACEMCO", "UPL", "WIPRO"] {allow-input: true}

view_company_forecast(selected_ticker, eval_results, show_plot=True)